**Overview**  
This notebook describes the process of tuning hyperparameters for NN acrhitecture, that we found earlier using *hyperopt* package. Also, we get final model for our task

First, we import all the necessary modules and define device for calculations - it is GPU card (*cuda:0*)

In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from functools import partial

from sklearn.model_selection import KFold, train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
device = 'cuda:0'
from warnings import filterwarnings
from hyperopt import hp, tpe, Trials, fmin, STATUS_OK

Get an absolute path, which will be helpful for training function

In [2]:
abs_path = os.path.abspath('')

This function just loads the data in forms of X and y DataFrames. It also loads "groups: file, that contains information about chemical nature of each instance in Dataset Then we check, that everything works correctly.

In [3]:
def load_data(path = abs_path, type = 'standard'):
    with open('y.pickle', 'rb') as inp:
        y = pickle.load(inp)
    if type == 'standard':
        with open('X_standard_dropped.pickle', 'rb') as inp:
            X= pickle.load(inp)
    elif type == 'padel':
        with open('X_padel_dropped.pickle', 'rb') as inp:
            X = pickle.load(inp)
    with open('groups.pickle', 'rb') as inp:
        groups = pickle.load(inp)
    return X, y, groups      
    
        

In [4]:
X, y, groups = load_data()

In [5]:
X.head()

,CTAB concentration (mM),Additive concentration,CTAB/additive,Temperature,fp0,fp1,fp2,fp3,fp4,fp5,...,CalcWHIM_20,CalcWHIM_21,CalcWHIM_30,CalcWHIM_41,CalcWHIM_42,CalcWHIM_53,CalcWHIM_63,CalcWHIM_91,CalcWHIM_101,CalcWHIM_102
0,20.0,3.245562,6.162262,50.0,0,0,0,0,0,0,...,0.132,0.114,0.552,0.623,0.485,0.285,0.596,0.179,0.291,0.333
1,20.0,7.526631,2.657232,50.0,0,0,0,0,0,0,...,0.132,0.114,0.552,0.623,0.485,0.285,0.596,0.179,0.291,0.333
2,20.0,8.556309,2.337456,50.0,0,0,0,0,0,0,...,0.132,0.114,0.552,0.623,0.485,0.285,0.596,0.182,0.291,0.333
3,20.0,9.569015,2.090079,50.0,0,0,0,0,0,0,...,0.132,0.114,0.552,0.623,0.485,0.285,0.596,0.182,0.291,0.333
4,20.0,10.568348,1.892443,50.0,0,0,0,0,0,0,...,0.132,0.114,0.552,0.623,0.485,0.285,0.596,0.179,0.291,0.333


In [6]:
y.head()

0    0
1    0
2    0
3    1
4    1
Name: Is gel, dtype: int64

Here we define classification NN class - the best NN, that we found on the previous step. Again, the first parameter is the number of descriptors, variable parameters are *n1*, *n2* and *n3* - number of neurons on each layer.

In [7]:
class Classification_NN(nn.Module):
    def __init__(self, n_descriptors, n1, n2, n3):
        super().__init__()
        self.linear_1 = nn.Linear(n_descriptors, n1)
        self.a1 = nn.Mish()
        self.dropout_1 = nn.Dropout(p = 0.3)
        self.linear_2 = nn.Linear(n1, n2)
        self.a2 = nn.Mish()

        self.linear_3 = nn.Linear(n2, n3)
        self.a3 = nn.Mish()
        
        self.linear_4 = nn.Linear(n3, 2)
    def forward(self, x):
        x = self.a1(self.linear_1(x))
        x = self.dropout_1(x)
        x = self.a2(self.linear_2(x))
        x = self.a3(self.linear_3(x))
        x = self.linear_4(x)
        return x

On the next two steps we create two helpful functions, that are used to facilitate fine-tuning of hyperparameters.

The first functions is used to train the model. It takes dataloader for training and perform the training loop for given number of epochs. The final result is the trained model

In [8]:
def train_model(train_dl, model, lr = 0.001, epochs = 300):
    model = model
    optimizer = torch.optim.Adam(params = model.parameters(), lr = lr, weight_decay = 1e-4)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
            model.train()
            for X_b, y_b in train_dl:
                X_b = X_b.to(device)
                y_b = y_b.to(device)
                output = model(X_b)
                loss = loss_fn(output, y_b)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    return model

The nxt function is used to evaluate the model. It takes dataloader for the standard evaluation loop for all batches in dataloader. On this step we asess F1-score, precition, recall and accuracy

In [9]:
def evaluate_model(test_dl, model):
    valid_f1, valid_precision, valid_recall, valid_accuracy = 0, 0, 0, 0
    for X_b, y_b in test_dl:
        X_b = X_b.to(device)
        y_b = y_b.to(device)
        output = model(X_b)
        out_labels = torch.argmax(output, dim = 1).cpu().detach().numpy()
        valid_f1 += f1_score(out_labels, y_b.cpu().detach().numpy(), zero_division = 0)
        valid_precision += precision_score(out_labels, y_b.cpu().detach().numpy(), zero_division = 0)
        valid_recall += recall_score(out_labels, y_b.cpu().detach().numpy(), zero_division = 0)
        valid_accuracy += accuracy_score(out_labels, y_b.cpu().detach().numpy())
    valid_f1 /= len(test_dl)
    valid_precision /= len(test_dl)
    valid_recall /= len(test_dl)
    valid_accuracy /= len(test_dl)
    return {'f1':valid_f1, 'precision':valid_precision, 'recall':valid_recall, 'accuracy':valid_accuracy}

Thefunction *cross-valid-NN* is a key to all the work done. As there is no convenient cross-validation method for PyTorch, such as implemented in Sklearn, we create our own function to perform it. It takes several steps:
1. Loads data in form of Pandas DataFrame datasets and splits it into main (X, y, groups) and "untouchable" test (X_test, y_test, groups_test) datasets
2. Makes a StratifiedKfold;
3. Creates empty lists to store values for classification metrics for each fold;
4. Then it performs trainign and evaluation for each fold in KFold, using two functions, that were defined above;
5. Metric for each fold are stored in lists;
6. Function returns mean value of lists for each metric.

It should be noted, that this function also uses type of deiscriptors - Standatd or PaDEL. This parameter defines the number of neurons on the input layer of NN


In [10]:
def cross_valid_NN(type, path = abs_path, lr = 0.001, n1 = 2048, n2 = 2048, n3 = 2048, ):
    X, y, groups = load_data(type = type)
    X, X_test, y, y_test, groups, groups_test = train_test_split(X, y, groups, random_state=0)
    
    kfold = StratifiedGroupKFold(n_splits = 5, shuffle = True, random_state = 0)
    f1_valid_history  = [] 
    accuracy_valid_history = []
    precision_valid_history = []
    recall_valid_history = []
    torch.manual_seed(0)
    for split in kfold.split(X, y, groups = groups):
        X_train, y_train = (X.iloc[split[0]], y.iloc[split[0]])
        X_test, y_test = (X.iloc[split[1]], y.iloc[split[1]])
        #scaling all data
        scaler = MinMaxScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        train_dl = DataLoader(TensorDataset(torch.tensor(X_train, dtype = torch.float32), torch.tensor(y_train.to_numpy(), dtype = torch.long)), batch_size = 20)
        test_dl = DataLoader(TensorDataset(torch.tensor(X_test, dtype = torch.float32), torch.tensor(y_test.to_numpy(), dtype = torch.long)), batch_size = 20)  
        if type == 'standard':
            model = Classification_NN(n_descriptors = 1871, 
                                      n1 = n1,
                                      n2 = n2,
                                      n3 = n3)
        elif type == 'padel':
            model = Classification_NN(n_descriptors = 1286, 
                                      n1 = n1,
                                      n2 = n2,
                                      n3 = n3)
            
        
        model.to(device)
        model = train_model(train_dl, model, lr)
        #model evaluation for every K-fold     
        model.eval()
        evaluation_results = evaluate_model(test_dl, model)
        f1_valid_history.append(evaluation_results['f1'])
        precision_valid_history.append(evaluation_results['precision'])
        accuracy_valid_history.append(evaluation_results['accuracy'])
        recall_valid_history.append(evaluation_results['recall'])
        
    return np.mean(accuracy_valid_history), np.mean(f1_valid_history), np.mean(recall_valid_history), np.mean(precision_valid_history)




    

This function is used to get best parameters of the NN, that we defined above. It used hyperopt functions *fmin* to minimize the loss, which, in this case, is the **accuracy** of classification. During the search for optimal architecture and fine-tuning we found, that using **precision** as an loss, as we did for shallow ML methods, lead to the high rejection rate, which severely decreases accuracy to about 50%. 

In [11]:
def get_best_params(search_space = {'n1':hp.randint('n1', 5000),
                 'n2':hp.randint('n2', 5000), 
                 'n3':hp.randint('n3', 5000), 
                 'lr':hp.uniform(label = 'lr', low = 10**(-5), high = 10**(-2))}, type = 'standard', max_evals = 50):
    def objective(params, type):
        score = cross_valid_NN(type = type, n1 = params['n1'], n2 = params['n2'], n3 = params['n3'], lr = params['lr'])
        return {'loss':-score[0], 'status':STATUS_OK, 'accuracy':score[0], 'f1':score[1], 'recall':score[2], 'precision':score[3]}
    trials = Trials()
    res = fmin(partial(objective, type = type), space = search_space, algo = tpe.suggest, trials = trials, max_evals = max_evals)
    return res, trials

And here we get best parameters for Standard and PaDEL datasets

In [12]:
result_standard = get_best_params()
best_params_standard = result_standard[0]

100%|██████████| 50/50 [2:11:22<00:00, 157.65s/trial, best loss: -0.6952807017543859]  


Learning rate, *n1*, *n2* and *n3*

In [16]:
print(best_params_standard)
with open('best_params_standard.pickle', 'wb') as out:
    pickle.dump(best_params_standard, out)


{'lr': 9.535938869109639e-05, 'n1': 4576, 'n2': 932, 'n3': 3968}


In [17]:
result_padel = get_best_params()
best_params_padel = result_padel[0]

100%|██████████| 50/50 [2:16:43<00:00, 164.07s/trial, best loss: -0.7021783625730994] 


In [18]:
print(best_params_padel)
with open('best_params_padel.pickle', 'wb') as out:
    pickle.dump(best_params_padel, out)

{'lr': 0.0009223316152169106, 'n1': 602, 'n2': 3076, 'n3': 812}


The function *final_training_and_evaluation* was created, first, to assess the NN, which was trained using the optimal hyperparameter. For this task we use test dataset, which was not employed duting cross-validation. The second task was to finally train the NN with the best hyperparameters and get the instance of it. Again, the evaluation is performed using two types, which are Standard and PaDEL descriptors

In [19]:
def final_training_and_evaluation(best_params, type, final = True):
    X, y, groups = load_data(type = type)
    if final:
        scaler = MinMaxScaler()
        X = scaler.fit_transform(X)
        dl = DataLoader(TensorDataset(torch.tensor(X, dtype = torch.float32), torch.tensor(y.to_numpy(), dtype = torch.long)), batch_size = 20)
    else:
        X, X_test, y, y_test = train_test_split(X, y, random_state = 0)
        scaler = MinMaxScaler()
        X = scaler.fit_transform(X)
        X_test = scaler.transform(X_test)
        dl = DataLoader(TensorDataset(torch.tensor(X, dtype = torch.float32), torch.tensor(y.to_numpy(), dtype = torch.long)), batch_size = 20)
        test_dl = DataLoader(TensorDataset(torch.tensor(X_test, dtype = torch.float32), torch.tensor(y_test.to_numpy(), dtype = torch.long)), batch_size = 20)
    
    if type == 'standard':
        model = Classification_NN(n_descriptors=1871, n1 = best_params['n1'], n2 = best_params['n2'], n3 = best_params['n3'])
    elif type == 'padel':
        model = Classification_NN(n_descriptors=1286, n1 = best_params['n1'], n2 = best_params['n2'], n3 = best_params['n3'])
    model.to(device)
    model = train_model(dl, model, best_params['lr'], epochs = 50)
    if final:
        return model, scaler
    else:
        return(evaluate_model(test_dl, model))

Here we have metrics for Standard dataset

In [20]:
scores_standard = final_training_and_evaluation(best_params=best_params_standard, type = 'standard', final = False)

In [21]:
scores_standard

{'f1': 0.7442270325022041,
 'precision': 0.8446338383838384,
 'recall': 0.6782051282051282,
 'accuracy': 0.7041666666666667}

In [23]:
scores_padel = final_training_and_evaluation(best_params=best_params_padel, type = 'padel', final = False)

In [24]:
scores_padel

{'f1': 0.7139127272422467,
 'precision': 0.8255366161616161,
 'recall': 0.6393611596736597,
 'accuracy': 0.6645833333333334}

As we can see, the best model uses Standard descriptors. Therefore, we save it state dict for further use

In [27]:
model_standard = final_training_and_evaluation(best_params_standard, type = 'standard', final = True)

In [29]:
torch.save(model_standard[0].state_dict(), 'model_standard_dict_state.pth')

In [30]:
with open('scaler.pickle', 'wb') as out:
    pickle.dump(model_standard[1], out)